# 🚢 تجارت‌یار — اجرای سریع روی Google Colab

این دفترچه سامانه **تجارت‌یار** را در **کمتر از ~۱ دقیقه** روی Colab بالا می‌آورد و یک **لینک عمومی موقت** می‌دهد.

**نکته‌ی مهم:** خروجی build (`dist`) از قبل آماده و در مخزن قرار دارد؛ بنابراین **نیازی به `npm install` و `npm run build` نیست** — فقط Node نصب و سرور اجرا می‌شود.

**هر سلول را به ترتیب با `Ctrl+Enter` اجرا کنید** (یا `Runtime → Run all`):
1. نصب Node.js ۲۰ (سریع)
2. دریافت کد از GitHub
3. اجرای سرور (بدون install/build)
4. بررسی سلامت
5. دانلود cloudflared
6. راه‌اندازی تونل (بلافاصله تمام می‌شود)
7. دریافت لینک دسترسی

> سلول‌های ۱ تا ۵ با `%%bash` و سلول‌های ۶ و ۷ با پایتون اجرا می‌شوند.
> لینک نهایی در خروجی **سلول ۷** چاپ می‌شود.

In [1]:
%%bash
node -v 2>/dev/null | grep -q "^v1[89]" && echo "node already present: $(node -v)" || (curl -fsSL https://nodejs.org/dist/v20.18.1/node-v20.18.1-linux-x64.tar.xz -o /tmp/node.txz && tar -xJf /tmp/node.txz -C /usr/local --strip-components=1 && echo "node installed: $(node -v)")

node installed: v20.18.1


In [2]:
%%bash
cd /content && rm -rf Tejaratyarr && git clone --depth 1 --branch arena/01a04f8e-tejaratyarr https://github.com/Setayesh-Jafari/Tejaratyarr.git

Cloning into 'Tejaratyarr'...


In [ ]:
%%bash
cd /content/Tejaratyarr && NODE_ENV=production setsid nohup node dist/server.cjs > /content/server.log 2>&1 < /dev/null &

In [ ]:
%%bash
sleep 4; curl -s http://localhost:3000/api/health; echo

In [ ]:
%%bash
cd /content/Tejaratyarr && (test -x cloudflared || curl -L --progress-bar -o cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64) && chmod +x cloudflared && ./cloudflared --version

In [ ]:
import subprocess

# بستن هر نمونه‌ی قبلی cloudflared (اگر کاربر سلول را دوباره اجرا کند)
subprocess.run("pkill -f 'cloudflared tunnel' || true", shell=True)

p = subprocess.Popen(
    ["./cloudflared", "tunnel", "--url", "http://localhost:3000",
     "--no-autoupdate", "--logfile", "/content/cloudflared.log"],
    cwd="/content/Tejaratyarr",
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
    stdin=subprocess.DEVNULL,
    start_new_session=True,
)
print("cloudflared started (PID", p.pid, ") — go to the next cell to get the link.")


In [ ]:
import time, re

print("waiting for the tunnel URL (usually 10-30 seconds)...")
url = None
for _ in range(45):
    try:
        with open("/content/cloudflared.log", "r", errors="ignore") as f:
            log = f.read()
        m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", log)
        if m:
            url = m.group(0)
            break
    except FileNotFoundError:
        pass
    time.sleep(2)

if url:
    print()
    print("LINK:", url)
else:
    print()
    print("link not ready yet — last lines of cloudflared log:")
    try:
        with open("/content/cloudflared.log", "r", errors="ignore") as f:
            print(f.read()[-2000:])
    except Exception as e:
        print(e)


## 📌 نکات

- لینک `trycloudflare` **موقت** است و تا زمانی که Colab روشن بماند کار می‌کند.
- اگر لینک چاپ نشد: سلول ۶ را دوباره اجرا کنید، سپس سلول ۷ را اجرا کنید. اگر باز هم نشد، سلول ۴ را اجرا کنید تا مطمئن شوید سرور بالا است.
- داده‌ها در Colab موقتی است؛ برای استفاده‌ی واقعی روی سرور خودتان اجرا کنید.
- فعال‌سازی هوش مصنوعی Gemini: قبل از اجرای سرور (سلول ۳)، `GEMINI_API_KEY` را تنظیم کنید.
- اجرای محلی (با کد منبع): `npm install` سپس `npm run dev` (پورت ۳۰۰۰).
- اجرای تولید محلی: `npm run build` سپس `NODE_ENV=production node dist/server.cjs`.

## 🛠 رفع اشکال
- **خطای `SyntaxError` در سلول bash؟** یعنی خط اول سلول `%%bash` نیست.
- **سرور بالا نیامد؟** سلول ۴ را اجرا کنید و اگر `curl` خطا داد، سلول ۳ را دوباره اجرا کنید.